In [147]:
# Sarah Sullivan
# April 7, 2026 


#Household Rosters

In [148]:
import numpy as np
import pandas as pd

In [149]:
root = "/Users/sarsul/Library/CloudStorage/Dropbox-UniversityofMichigan/Sarah Sullivan/SARAH-SOFT/research/psid/"

df = pd.read_stata(root + "_psid_out_v2.dta", convert_categoricals=False)

In [150]:
keep_vars = ["fam", "ID", "yr", "family_year_id", "hhr", "ages_hhr", "rel_hhr", "age_", "age_first_observed", "waves", "waves_18_under"]
keep = ["ID_aM", "ID_aD", "ID_bM", "ID_bD"]
gpar_cols = ['ID_aM_aM', 'ID_aM_aD', 'ID_aM_bM', 'ID_aM_bD', 'ID_aD_aM', 'ID_aD_aD', 'ID_aD_bM', 'ID_aD_bD', 'ID_bM_aM', 'ID_bM_aD', 'ID_bM_bM', 'ID_bM_bD', 'ID_bD_aM', 'ID_bD_aD', 'ID_bD_bM', 'ID_bD_bD']
sibs= ["ID_S01", "ID_S02", "ID_S03", "ID_S04", "ID_S05", "ID_S06", "ID_S07", "ID_S08", "ID_S09", "ID_S10", "ID_S11", "ID_S12", "ID_S13", "ID_S14", "ID_S15", "ID_S16"]
df = df[keep_vars + keep + gpar_cols + sibs]


In [151]:
# Convert selected ID/time columns from float to integer
int_cols = ["fam", "ID", "yr", "family_year_id"]
df[int_cols] = df[int_cols].astype("int64")


In [152]:
def parse_tuple_string(x):
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []

    s = str(x).strip()
    if s == "" or s.lower() == "none":
        return []

    out = []
    for tok in s.split():
        if tok == ".":
            continue
        try:
            out.append(int(float(tok)))
        except ValueError:
            out.append(tok)
    return out


In [153]:
df['hhr'] = df['hhr'].apply(parse_tuple_string)
df['ages_hhr'] = df['ages_hhr'].apply(parse_tuple_string)
df['rel_hhr'] = df['rel_hhr'].apply(parse_tuple_string)

In [154]:
def remove_self_from_hhr_and_ages(row):
    hhr = row["hhr"] if isinstance(row["hhr"], list) else []
    ages = row["ages_hhr"] if isinstance(row["ages_hhr"], list) else []
    self_id = int(row["ID"])

    new_hhr = []
    new_ages = []

    for i, pid in enumerate(hhr):
        try:
            pid_int = int(float(pid))
        except (TypeError, ValueError):
            pid_int = pid

        if pid_int == self_id:
            continue

        new_hhr.append(pid)
        if i < len(ages):
            new_ages.append(ages[i])

    return pd.Series({"hhr": new_hhr, "ages_hhr": new_ages})

df[["hhr", "ages_hhr"]] = df.apply(remove_self_from_hhr_and_ages, axis=1)

In [155]:
# within-person previous roster (ordered by year)
df = df.sort_values(["ID", "yr"]).copy()

df["hhr_prev"] = df.groupby("ID")["hhr"].shift(1)
df["ages_prev"] = df.groupby("ID")["ages_hhr"].shift(1)
df["rel_prev"] = df.groupby("ID")["rel_hhr"].shift(1)

# ensure list-like values
df["hhr_prev"] = df["hhr_prev"].apply(parse_tuple_string)
df['ages_prev'] = df['ages_prev'].apply(parse_tuple_string)
df['rel_prev'] = df['rel_prev'].apply(parse_tuple_string)

# ids of who left and who came in each year (within person)
df["who_left"] = df.apply(
    lambda row: [x for x in row["hhr_prev"] if x not in row["hhr"]],
    axis=1
)
df["who_came"] = df.apply(
    lambda row: [x for x in row["hhr"] if x not in row["hhr_prev"]],
    axis=1
)

In [156]:
# Set who_came to empty list for each person's first observed year
# Set who_came to empty list for each person's first observed year
first_obs_mask = df["ID"].ne(df["ID"].shift(1))


df.loc[first_obs_mask, "who_came"] = pd.Series(
    [[] for _ in range(first_obs_mask.sum())],
    index=df.index[first_obs_mask]
)



In [157]:
last_obs_mask = df["ID"].ne(df["ID"].shift(-1))

df.loc[last_obs_mask, "who_left"] = pd.Series(
    [[] for _ in range(last_obs_mask.sum())],
    index=df.index[last_obs_mask]
)

In [158]:
def get_ages_left(row):
    hhr_prev = row['hhr_prev'] if isinstance(row['hhr_prev'], list) else []
    ages_prev = row['ages_prev'] if isinstance(row['ages_prev'], list) else []
    who_left = row['who_left'] if isinstance(row['who_left'], list) else []
    return [ages_prev[hhr_prev.index(pid)] for pid in who_left if pid in hhr_prev and hhr_prev.index(pid) < len(ages_prev)]

def get_ages_came(row):
    hhr = row['hhr'] if isinstance(row['hhr'], list) else []
    ages = row['ages_hhr'] if isinstance(row['ages_hhr'], list) else []
    who_came = row['who_came'] if isinstance(row['who_came'], list) else []
    return [ages[hhr.index(pid)] for pid in who_came if pid in hhr and hhr.index(pid) < len(ages)]

In [159]:
df['ages_left'] = df.apply(get_ages_left, axis=1)
df['ages_came'] = df.apply(get_ages_came, axis=1)

df['adult_came'] = df['ages_came'].apply(
    lambda ages: int(isinstance(ages, list) and any(age >= 18 for age in ages))
)

df['child_came'] = df['ages_came'].apply(
    lambda ages: int(isinstance(ages, list) and any(age < 18 for age in ages))
)

df['adult_left'] = df['ages_left'].apply(
    lambda ages: int(isinstance(ages, list) and any(age >= 18 for age in ages))
)

df['child_left'] = df['ages_left'].apply(
    lambda ages: int(isinstance(ages, list) and any(age < 18 for age in ages))
)

df['n_adults_left'] = df['ages_left'].apply(
    lambda ages: sum(age >= 18 for age in ages) if isinstance(ages, list) else 0
)

df['n_adults_came'] = df['ages_came'].apply(
    lambda ages: sum(age >= 18 for age in ages) if isinstance(ages, list) else 0
)

In [160]:
# Create sib_list: list of all sibling IDs (ID_S01 through ID_S16)
sib_cols = [f'ID_S{str(i).zfill(2)}' for i in range(1, 17)]
df['sib_list'] = df[sib_cols].apply(lambda row: [x for x in row if pd.notna(x)], axis=1)

In [161]:
df['sib_came'] = df.apply(
    lambda row: int(isinstance(row['who_came'], list) and isinstance(row['sib_list'], list) and any(id in row['sib_list'] for id in row['who_came'])),
    axis=1
)

df['sib_left'] = df.apply(
    lambda row: int(isinstance(row['who_left'], list) and isinstance(row['sib_list'], list) and any(id in row['sib_list'] for id in row['who_left'])),
    axis=1
)   

In [162]:
# Get ages of siblings who came
def get_sib_ages_came(row):
    if not isinstance(row['who_came'], list) or not isinstance(row['sib_list'], list):
        return []
    hhr = row['hhr'] if isinstance(row['hhr'], list) else []
    ages = row['ages_hhr'] if isinstance(row['ages_hhr'], list) else []
    # Get IDs that are both in who_came and sib_list
    sibs_who_came = [id for id in row['who_came'] if id in row['sib_list']]
    # Get ages for those siblings
    return [ages[hhr.index(sib_id)] for sib_id in sibs_who_came if sib_id in hhr and hhr.index(sib_id) < len(ages)]

# Get ages of siblings who left
def get_sib_ages_left(row):
    if not isinstance(row['who_left'], list) or not isinstance(row['sib_list'], list):
        return []
    hhr_prev = row['hhr_prev'] if isinstance(row['hhr_prev'], list) else []
    ages_prev = row['ages_prev'] if isinstance(row['ages_prev'], list) else []
    # Get IDs that are both in who_left and sib_list
    sibs_who_left = [id for id in row['who_left'] if id in row['sib_list']]
    # Get ages for those siblings
    return [ages_prev[hhr_prev.index(sib_id)] for sib_id in sibs_who_left if sib_id in hhr_prev and hhr_prev.index(sib_id) < len(ages_prev)]

df['sib_ages_came'] = df.apply(get_sib_ages_came, axis=1)
df['sib_ages_left'] = df.apply(get_sib_ages_left, axis=1)

In [163]:
# Create par_list: list of ID's for parents
par_cols = ['ID_aM', 'ID_aD', 'ID_bM', 'ID_bD']
df['par_list'] = df[par_cols].apply(lambda row: [x for x in row if pd.notna(x)], axis=1)

In [164]:
df['par_came'] = df.apply(
    lambda row: int(isinstance(row['who_came'], list) and isinstance(row['par_list'], list) and any(id in row['par_list'] for id in row['who_came'])),
    axis=1
)

df['par_left'] = df.apply(
    lambda row: int(isinstance(row['who_left'], list) and isinstance(row['par_list'], list) and any(id in row['par_list'] for id in row['who_left'])),
    axis=1
)

In [165]:
gpar_cols = ['ID_aM_aM', 'ID_aM_aD', 'ID_aM_bM', 'ID_aM_bD', 'ID_aD_aM', 'ID_aD_aD', 'ID_aD_bM', 'ID_aD_bD', 'ID_bM_aM', 'ID_bM_aD', 'ID_bM_bM', 'ID_bM_bD', 'ID_bD_aM', 'ID_bD_aD', 'ID_bD_bM', 'ID_bD_bD']
df['gpar_list'] = df[gpar_cols].apply(lambda row: [x for x in row if pd.notna(x)], axis=1)

In [166]:
df['gpar_came'] = df.apply(
    lambda row: int(isinstance(row['who_came'], list) and isinstance(row['gpar_list'], list) and any(id in row['gpar_list'] for id in row['who_came'])),
    axis=1
)

df['gpar_left'] = df.apply(
    lambda row: int(isinstance(row['who_left'], list) and isinstance(row['gpar_list'], list) and any(id in row['gpar_list'] for id in row['who_left'])),
    axis=1
)

In [167]:
new_keep = ["fam", "ID", "yr", "family_year_id", "hhr", "ages_hhr", "rel_hhr", "age_", "age_first_observed", "who_left", "who_came", "ages_left", "ages_came", "adult_came", "adult_left", "child_came", "child_left", "n_adults_left", "n_adults_came", "sib_came", "sib_left", "par_came", "par_left", "gpar_came", "gpar_left", 'sib_ages_came', 'sib_ages_left']
df2 = df.copy()[new_keep]
df2

,fam,ID,yr,family_year_id,hhr,ages_hhr,rel_hhr,age_,age_first_observed,who_left,...,n_adults_left,n_adults_came,sib_came,sib_left,par_came,par_left,gpar_came,gpar_left,sib_ages_came,sib_ages_left
0,1,1030,1973,17,"[1003, 1004]","[25, 23]","[1, 2, 3]",1.0,1.0,[],...,0,0,0,0,0,0,0,0,[],[]
1,1,1030,1974,377,"[1003, 1004]","[27, 25]","[1, 2, 3]",1.0,1.0,[],...,0,0,0,0,0,0,0,0,[],[]
2,1,1030,1975,2848,"[1003, 1004]","[28, 26]","[1, 2, 3]",3.0,1.0,[],...,0,0,0,0,0,0,0,0,[],[]
3,1,1030,1976,2425,"[1003, 1004]","[30, 28]","[1, 2, 3]",4.0,1.0,[],...,0,0,0,0,0,0,0,0,[],[]
4,1,1030,1977,2012,"[1003, 1004]","[30, 28]","[1, 2, 3]",5.0,1.0,[],...,0,0,0,0,0,0,0,0,[],[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
325045,9308,9308003,1995,8660,"[9308001, 9308002, 9308004, 9308170]","[28, 9, 51]","[90, 10, 30, 30, 50]",9.0,7.0,[],...,0,1,0,0,0,0,0,0,[],[]
325046,9308,9308004,1992,9496,"[9308002, 9308003]","[25, 7]","[10, 30, 30]",3.0,3.0,[],...,0,0,0,0,0,0,0,0,[],[]
325047,9308,9308004,1993,4839,"[9308002, 9308003]","[26, 8]","[10, 30, 30]",3.0,3.0,[],...,0,0,0,0,0,0,0,0,[],[]
325048,9308,9308004,1994,14295,"[9308002, 9308003]","[27, 9]","[10, 30, 30]",4.0,3.0,[],...,0,0,0,0,0,0,0,0,[],[]


In [168]:
df2['any_change'] = df2.apply(
    lambda row: int(len(row['who_came']) > 0 or len(row['who_left']) > 0),
    axis=1
)

# Mark any_change = 1 for all rows of an ID if any row has a change
df2['any_change'] = df2.groupby('ID')['any_change'].transform('max')
df2

,fam,ID,yr,family_year_id,hhr,ages_hhr,rel_hhr,age_,age_first_observed,who_left,...,n_adults_came,sib_came,sib_left,par_came,par_left,gpar_came,gpar_left,sib_ages_came,sib_ages_left,any_change
0,1,1030,1973,17,"[1003, 1004]","[25, 23]","[1, 2, 3]",1.0,1.0,[],...,0,0,0,0,0,0,0,[],[],0
1,1,1030,1974,377,"[1003, 1004]","[27, 25]","[1, 2, 3]",1.0,1.0,[],...,0,0,0,0,0,0,0,[],[],0
2,1,1030,1975,2848,"[1003, 1004]","[28, 26]","[1, 2, 3]",3.0,1.0,[],...,0,0,0,0,0,0,0,[],[],0
3,1,1030,1976,2425,"[1003, 1004]","[30, 28]","[1, 2, 3]",4.0,1.0,[],...,0,0,0,0,0,0,0,[],[],0
4,1,1030,1977,2012,"[1003, 1004]","[30, 28]","[1, 2, 3]",5.0,1.0,[],...,0,0,0,0,0,0,0,[],[],0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
325045,9308,9308003,1995,8660,"[9308001, 9308002, 9308004, 9308170]","[28, 9, 51]","[90, 10, 30, 30, 50]",9.0,7.0,[],...,1,0,0,0,0,0,0,[],[],1
325046,9308,9308004,1992,9496,"[9308002, 9308003]","[25, 7]","[10, 30, 30]",3.0,3.0,[],...,0,0,0,0,0,0,0,[],[],1
325047,9308,9308004,1993,4839,"[9308002, 9308003]","[26, 8]","[10, 30, 30]",3.0,3.0,[],...,0,0,0,0,0,0,0,[],[],1
325048,9308,9308004,1994,14295,"[9308002, 9308003]","[27, 9]","[10, 30, 30]",4.0,3.0,[],...,0,0,0,0,0,0,0,[],[],1


In [169]:
df2["birth_year"] = (df2["yr"] - df2["age_"]).astype(int)
df2["birth_year"] = df2.groupby("ID")["birth_year"].transform("min").astype(int)

start_year = 1948

# 5-yr cohorts
bin_size = 5

# Create bins from 1948 to just past 2020
bins_5 = range(start_year, 2026, bin_size)

# Create labels like "1948-1953", "1954-1959", etc.
labels_5 = [f"{y}-{y + bin_size - 1}" for y in bins_5[:-1]]

# Assign 5-year cohorts
df2['cohort_5'] = pd.cut(
    df2['birth_year'],
    bins=list(bins_5),
    labels=labels_5,
    right=False  # intervals are [left, right), so 1948 <= x < 1953
)

# 10-yr cohorts
bin_size = 10

# Create bins from 1948 to just past 2020
bins_10 = range(start_year, 2026, bin_size)

# Create labels like "1948-1957", "1958-1967", etc.
labels_10 = [f"{y}-{y + bin_size - 1}" for y in bins_10[:-1]]

# Assign 10-year cohorts
df2['cohort_10'] = pd.cut(
    df2['birth_year'],
    bins=list(bins_10),
    labels=labels_10,
    right=False  # intervals are [left, right), so 1948 <= x < 1958
)


In [172]:
df2["hh_came"] = df2["who_came"].apply(lambda x: int(isinstance(x, list) and len(x) > 0))
df2["hh_left"] = df2["who_left"].apply(lambda x: int(isinstance(x, list) and len(x) > 0))


In [173]:
df2.to_csv("full_sample_changes.csv", index=False)

In [174]:
df3 = df2[df2['any_change'] == 1].copy()
df3.to_csv("only_changed_sample.csv", index=False)